# Earthquake Catalog Metadata Parser

This notebook parses MLDD (Machine Learning Double Difference) earthquake catalog files into structured pandas DataFrames for further analysis.

In [ ]:
import numpy as np
import pandas as pd
from datetime import datetime
import obspy
import splitting_functions

In [ ]:
catalog_file = "../data/Axial.MLDD.v202112.2"

In [ ]:
def parse_mldd_catalog(filename):
    """
    Parse MLDD earthquake catalog file into a pandas DataFrame.
    
    Parameters:
    -----------
    filename : str
        Path to the MLDD catalog file
        
    Returns:
    --------
    pd.DataFrame
        DataFrame containing parsed earthquake catalog with columns:
        - datetime: event origin time
        - year, month, day, hour, minute, second: time components
        - latitude, longitude, depth_km: hypocenter location
        - err_x, err_y, err_z: location uncertainties (km)
        - num_phases: number of phase picks
        - rms: root mean square residual (seconds)
        - magnitude: event magnitude
        - event_id: unique event identifier
    """
    events = []
    
    with open(filename, 'r') as f:
        for line in f:
            parts = line.split()
            
            # Parse date and time
            year = int(parts[0])
            month = int(parts[1])
            day = int(parts[2])
            hour = int(parts[3])
            minute = int(parts[4])
            second = float(parts[5])

            # Create datetime using timestamp approach
            # Handle edge case where second >= 60 (rounding)
            from datetime import datetime, timezone, timedelta
            second_int = int(second)
            microseconds = int((second - second_int) * 1e6)
            
            if second_int >= 60:
                # Add extra seconds to minutes
                extra_minutes = second_int // 60
                second_int = second_int % 60
                dt_base = datetime(year, month, day, hour, minute, second_int, tzinfo=timezone.utc)
                dt_py = dt_base.replace(microsecond=microseconds) + timedelta(minutes=extra_minutes)
            else:
                dt_base = datetime(year, month, day, hour, minute, second_int, tzinfo=timezone.utc)
                dt_py = dt_base.replace(microsecond=microseconds)
            
            # Convert to ObsPy UTCDateTime
            dt = obspy.UTCDateTime(dt_py)
            
            # Parse location and uncertainties
            latitude = float(parts[6])
            longitude = float(parts[7])
            depth_km = float(parts[8])
            eh1 = float(parts[9])
            eh2 = float(parts[10])
            az = int(parts[11])
            ev = float(parts[12])
            magnitude = float(parts[13])
            event_id = float(parts[14])
            
            events.append({
                'datetime': dt,
                'year': year,
                'month': month,
                'day': day,
                'hour': hour,
                'minute': minute,
                'second': second,
                'latitude': latitude,
                'longitude': longitude,
                'depth_km': depth_km,
                'eh1': eh1,
                'eh2': eh2,
                'az': az,
                'ev': ev,
                'magnitude': magnitude,
                'event_id': event_id
            })
    
    df = pd.DataFrame(events)
    
    # Set event_id as index if available
    if 'event_id' in df.columns and not df['event_id'].isna().all():
        df = df.set_index('event_id')
    
    return df

In [ ]:
# Parse the catalog
catalog_df = parse_mldd_catalog(catalog_file)

# Display basic information
print(f"Total events in catalog: {len(catalog_df)}")
print(f"\nCatalog time range: {catalog_df['datetime'].min()} to {catalog_df['datetime'].max()}")
print(f"\nMagnitude range: {catalog_df['magnitude'].min():.2f} to {catalog_df['magnitude'].max():.2f}")
print(f"\nDepth range: {catalog_df['depth_km'].min():.2f} to {catalog_df['depth_km'].max():.2f} km")
print(f"\n" + "="*80)
print("\nFirst 10 events:")
print(catalog_df.head(10))

In [ ]:
# Save to CSV
output_file = "../data/mldd_catalog_parsed.csv"
catalog_df.to_csv(output_file)
print(f"Catalog saved to: {output_file}")
print(f"Total events saved: {len(catalog_df)}")

In [ ]:
# Load the phase file and parse it
phase_file = "../data/Axial.DD.pha.v20221129.1"

events_df, phases_df = splitting_functions.parse_phase_file(phase_file)

In [ ]:
base_dt = pd.to_datetime(events_df[['year', 'month', 'day', 'hour', 'minute']], utc=True)
events_df['datetime'] = (base_dt + pd.to_timedelta(events_df['second'], unit='s')).apply(lambda ts: obspy.UTCDateTime(ts.to_pydatetime()))

In [ ]:
# Sort events_df by event_datetime UTCDateTime in ascending order
events_df_sorted = events_df.sort_values(by='datetime')

In [ ]:
events_df = events_df_sorted

In [ ]:
phases_df

In [ ]:
# Sort phases_df by event_datetime UTCDateTime in ascending order
phases_df_sorted = phases_df.sort_values(by='event_datetime')

In [ ]:
phases_df = phases_df_sorted

In [ ]:
catalog_df

In [ ]:
# Integrate phases and catalog data
# Pivot phases to have P and S times as separate columns per station
phases_pivot = phases_df.pivot_table(
    index=['event_id', 'station'],
    columns='phase_type',
    values='arrival_time',
    aggfunc='first'
).reset_index()

# Rename columns for clarity
phases_pivot.columns.name = None
phases_pivot = phases_pivot.rename(columns={'P': 'p_arrival_time', 'S': 's_arrival_time'})

# Keep only stations with both P and S picks
phases_with_both = phases_pivot.dropna(subset=['p_arrival_time', 's_arrival_time'])

# Merge with catalog data
integrated_df = phases_with_both.merge(catalog_df, left_on='event_id', right_index=True, how='inner')

# Remove year, month, day, hour, minute, second columns
columns_to_drop = ['year', 'month', 'day', 'hour', 'minute', 'second']
integrated_df = integrated_df.drop(columns=columns_to_drop, errors='ignore')

print(f"Total event-station pairs with both P and S picks: {len(integrated_df)}")
print(f"\nColumns: {list(integrated_df.columns)}")
print(f"\n{integrated_df.head(10)}")

In [ ]:
# Integrate phases and catalog data
# Explicitly separate P and S picks, deduplicate per (event_id, station), then merge
p_picks = (phases_df[phases_df['phase_type'] == 'P']
           [['station', 'arrival_time', 'event_datetime']])

s_picks = (phases_df[phases_df['phase_type'] == 'S']
           [['station', 'arrival_time', 'event_datetime']])

# Inner join ensures only (event, station) pairs with both P and S picks are kept
phases_with_both = p_picks.merge(s_picks, on=['station', 'event_datetime'], how='inner', suffixes=('_p', '_s'))

# --- Timezone Correction for Merge ---
# This is an idempotent operation: it checks types before converting to avoid errors on re-runs.

# 1. Ensure phase 'event_datetime' is timezone-aware (UTC).
phases_with_both['event_datetime'] = pd.to_datetime(phases_with_both['event_datetime'])
if phases_with_both['event_datetime'].dt.tz is None:
    phases_with_both['event_datetime'] = phases_with_both['event_datetime'].dt.tz_localize('UTC')
else:
    phases_with_both['event_datetime'] = phases_with_both['event_datetime'].dt.tz_convert('UTC')

# 2. Ensure the target dataframe's datetime column is also UTC-aware.
phases_df['event_datetime'] = pd.to_datetime(phases_df['event_datetime'])
if phases_df['event_datetime'].dt.tz is None:
    phases_df['event_datetime'] = phases_df['event_datetime'].dt.tz_localize('UTC')
else:
    phases_df['event_datetime'] = phases_df['event_datetime'].dt.tz_convert('UTC')


# Merge the dataframe that has both P and S picks with the original phases_df
integrated_df = pd.merge(
    phases_with_both, 
    phases_df, 
    on=['station', 'event_datetime'],
    how='inner'
)


# Remove year, month, day, hour, minute, second columns
#columns_to_drop = ['year', 'month', 'day', 'hour', 'minute', 'second']
#integrated_df = integrated_df.drop(columns=columns_to_drop, errors='ignore')

print(f"Total event-station pairs with both P and S picks: {len(integrated_df)}")
print(f"\nColumns: {list(integrated_df.columns)}")
print(f"\n{integrated_df.head(10)}")

In [ ]:
integrated_df

In [ ]:
# Drop weight, phase_type, and quality columns, then remove the leading 'OO' from station names
columns_to_drop = ['weight', 'phase_type', 'quality']
integrated_df = integrated_df.drop(columns=columns_to_drop, errors='ignore')

# Remove leading 'OO' from station names if present
integrated_df['station'] = integrated_df['station'].str.replace(r'^OO', '', regex=True)

In [ ]:
integrated_df = integrated_df.drop(columns='arrival_time', errors='ignore')

In [ ]:
# Remove duplicate eventstation pairs if they exist (keep the first occurrence)
integrated_df = integrated_df.drop_duplicates(subset=['event_datetime', 'station', 'arrival_time_s', 'arrival_time_p', 'event_depth'], keep='first')

In [ ]:
integrated_df

In [ ]:
# Cut out all events before 2015-01-22, then reset index
cutoff_date = pd.Timestamp("2015-01-22T00:00:00Z")
integrated_df = integrated_df[integrated_df['event_datetime'] >= cutoff_date]
integrated_df = integrated_df.reset_index(drop=True)

In [ ]:
integrated_df = integrated_df.drop(columns='event_id')

In [ ]:
# Drop all events from AXID1 station
integrated_df = integrated_df[integrated_df['station'] != 'AXID1'].copy()

In [ ]:
integrated_df

In [ ]:
# Save integrated catalog
integrated_output = "../data/mldd_catalog_2015_2021.csv"
integrated_df.to_csv(integrated_output, index=False)
print(f"Integrated catalog saved to: {integrated_output}")
print(f"Total records: {len(integrated_df)}")